# Severity Modeling

This notebook focuses on severity estimation for plant health issues.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import sys
import os

# Add src to path
sys.path.append('../src')

from models import SeverityEstimationModel
from data_loader import get_data_loaders
from evaluate import SeverityEvaluator

# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

plt.rcParams['figure.figsize'] = (12, 8)

## Load Data

In [ ]:
# Load data loaders
data_loaders = get_data_loaders(
    data_dir='../data/raw',
    metadata_file='../data/raw/metadata.csv',
    batch_size=32,
    num_workers=4,
    image_size=224
)

print(f"Data loaders loaded successfully")

## Analyze Severity Distribution

In [ ]:
# Collect severity values from dataset
severity_values = []
for _, labels in data_loaders['train']:
    severity_values.extend(labels['severity'].numpy())

severity_values = np.array(severity_values)

print(f"Severity statistics:")
print(f"  Mean: {severity_values.mean():.3f}")
print(f"  Std: {severity_values.std():.3f}")
print(f"  Min: {severity_values.min():.3f}")
print(f"  Max: {severity_values.max():.3f}")
print(f"  Median: {np.median(severity_values):.3f}")

# Plot distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(severity_values, bins=30, color='#3498db', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Severity Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Severity Distribution')
axes[0].axvline(severity_values.mean(), color='red', linestyle='--', 
               label=f'Mean: {severity_values.mean():.2f}')
axes[0].legend()

axes[1].boxplot(severity_values)
axes[1].set_ylabel('Severity Score')
axes[1].set_title('Severity Boxplot')

plt.tight_layout()
plt.show()

## Severity vs Health Status Analysis

In [ ]:
# Analyze severity by health status
health_severity = {0: [], 1: [], 2: []}  # healthy, stressed, diseased

for _, labels in data_loaders['train']:
    health = labels['health_status'].numpy()
    severity = labels['severity'].numpy()
    
    for h, s in zip(health, severity):
        if h in health_severity:
            health_severity[h].append(s)

# Create boxplot
fig, ax = plt.subplots(figsize=(10, 6))

data_to_plot = [health_severity[0], health_severity[1], health_severity[2]]
bp = ax.boxplot(data_to_plot, labels=['Healthy', 'Stressed', 'Diseased'],
                patch_artist=True)

# Color the boxes
colors = ['#2ecc71', '#f39c12', '#e74c3c']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)

ax.set_ylabel('Severity Score')
ax.set_title('Severity Distribution by Health Status')
plt.tight_layout()
plt.show()

# Print statistics
print("\nSeverity by Health Status:")
for status, values in [('Healthy', health_severity[0]), 
                        ('Stressed', health_severity[1]),
                        ('Diseased', health_severity[2])]:
    if values:
        print(f"{status}: Mean={np.mean(values):.3f}, Std={np.std(values):.3f}, Count={len(values)}")

## Initialize Severity Model

In [ ]:
# Create severity estimation model
severity_model = SeverityEstimationModel(
    backbone='efficientnet_b0',
    pretrained=True,
    num_severity_levels=4
)

severity_model = severity_model.to(device)

# Count parameters
total_params = sum(p.numel() for p in severity_model.parameters())
trainable_params = sum(p.numel() for p in severity_model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## Test Forward Pass

In [ ]:
# Test forward pass
severity_model.eval()
with torch.no_grad():
    dummy_input = torch.randn(2, 3, 224, 224).to(device)
    
    # Regression mode
    reg_output = severity_model(dummy_input, mode='regression')
    print(f"Regression output shape: {reg_output.shape}")
    print(f"Regression output range: [{reg_output.min():.3f}, {reg_output.max():.3f}]")
    
    # Classification mode
    cls_output = severity_model(dummy_input, mode='classification')
    print(f"Classification output shape: {cls_output.shape}")

## Baseline Model Evaluation (Before Training)

In [ ]:
# Initialize evaluator
evaluator = SeverityEvaluator(severity_model, device=device)

# Evaluate baseline performance
baseline_metrics = evaluator.evaluate(data_loaders['test'])

print("Baseline Metrics (Untrained Model):")
print("="*40)
for key, value in baseline_metrics.items():
    print(f"{key:20s}: {value:.4f}")

## Training Setup

In [ ]:
# Setup training for severity model
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter

# Optimizer
optimizer = optim.AdamW(severity_model.parameters(), lr=1e-4, weight_decay=1e-5)

# Loss function (MSE for regression)
criterion = nn.MSELoss()

# Learning rate scheduler
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-6)

# Tensorboard writer
writer = SummaryWriter('../experiments/logs/severity')

print("Training setup complete")

## Training Loop (Uncomment to run)

In [ ]:
# Training loop
# num_epochs = 50
# best_val_loss = float('inf')
#
# for epoch in range(1, num_epochs + 1):
#     # Training
#     severity_model.train()
#     train_loss = 0
#     
#     for images, labels in data_loaders['train']:
#         images = images.to(device)
#         severity = labels['severity'].to(device)
#         
#         optimizer.zero_grad()
#         outputs = severity_model(images, mode='regression')
#         loss = criterion(outputs, severity)
#         
#         loss.backward()
#         torch.nn.utils.clip_grad_norm_(severity_model.parameters(), max_norm=1.0)
#         optimizer.step()
#         
#         train_loss += loss.item()
#     
#     train_loss /= len(data_loaders['train'])
#     
#     # Validation
#     severity_model.eval()
#     val_loss = 0
#     
#     with torch.no_grad():
#         for images, labels in data_loaders['val']:
#             images = images.to(device)
#             severity = labels['severity'].to(device)
#             
#             outputs = severity_model(images, mode='regression')
#             loss = criterion(outputs, severity)
#             val_loss += loss.item()
#     
#     val_loss /= len(data_loaders['val'])
#     
#     # Update learning rate
#     scheduler.step()
#     
#     # Log
#     writer.add_scalar('Loss/train', train_loss, epoch)
#     writer.add_scalar('Loss/val', val_loss, epoch)
#     writer.add_scalar('Learning_rate', optimizer.param_groups[0]['lr'], epoch)
#     
#     print(f"Epoch {epoch}/{num_epochs} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
#     
#     # Save best model
#     if val_loss < best_val_loss:
#         best_val_loss = val_loss
#         torch.save({
#             'epoch': epoch,
#             'model_state_dict': severity_model.state_dict(),
#             'optimizer_state_dict': optimizer.state_dict(),
#             'val_loss': val_loss
#         }, '../experiments/checkpoints/severity_best.pth')
#         print(f"  Saved best model with val_loss: {val_loss:.4f}")

## Load Trained Model (if available)

In [ ]:
# Load checkpoint if available
checkpoint_path = '../experiments/checkpoints/severity_best.pth'

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    severity_model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Model loaded from {checkpoint_path}")
    print(f"Validation loss: {checkpoint['val_loss']:.4f}")
else:
    print("No checkpoint found. Using untrained model for evaluation.")

## Detailed Evaluation

In [ ]:
# Collect predictions
severity_model.eval()
all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in data_loaders['test']:
        images = images.to(device)
        severity = labels['severity'].numpy()
        
        outputs = severity_model(images, mode='regression')
        predictions = outputs.cpu().numpy()
        
        all_predictions.append(predictions)
        all_labels.append(severity)

all_predictions = np.concatenate(all_predictions)
all_labels = np.concatenate(all_labels)

# Calculate metrics
mae = mean_absolute_error(all_labels, all_predictions)
rmse = np.sqrt(mean_squared_error(all_labels, all_predictions))
r2 = r2_score(all_labels, all_predictions)

print("Detailed Severity Metrics:")
print("="*40)
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Root Mean Square Error (RMSE): {rmse:.4f}")
print(f"R² Score: {r2:.4f}")

## Prediction vs Actual Plot

In [ ]:
# Plot predictions vs actual
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
axes[0].scatter(all_labels, all_predictions, alpha=0.5, s=20)
axes[0].plot([0, 3], [0, 3], 'r--', label='Perfect Prediction')
axes[0].set_xlabel('Actual Severity')
axes[0].set_ylabel('Predicted Severity')
axes[0].set_title('Predicted vs Actual Severity')
axes[0].legend()
axes[0].set_xlim(0, 3)
axes[0].set_ylim(0, 3)
axes[0].grid(True, alpha=0.3)

# Residual plot
residuals = all_predictions - all_labels
axes[1].scatter(all_labels, residuals, alpha=0.5, s=20)
axes[1].axhline(y=0, color='r', linestyle='--')
axes[1].set_xlabel('Actual Severity')
axes[1].set_ylabel('Residuals (Predicted - Actual)')
axes[1].set_title('Residual Plot')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Error Distribution Analysis

In [ ]:
# Analyze error distribution by severity level
severity_bins = [0, 1, 2, 3]
bin_labels = ['0-1', '1-2', '2-3']

errors_by_bin = []
for i in range(len(severity_bins) - 1):
    mask = (all_labels >= severity_bins[i]) & (all_labels < severity_bins[i + 1])
    if mask.sum() > 0:
        bin_errors = np.abs(all_predictions[mask] - all_labels[mask])
        errors_by_bin.append(bin_errors)
        print(f"Severity {bin_labels[i]}: MAE={bin_errors.mean():.4f}, Count={mask.sum()}")

# Plot error distribution
fig, ax = plt.subplots(figsize=(10, 6))
bp = ax.boxplot(errors_by_bin, labels=bin_labels, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('#3498db')

ax.set_ylabel('Absolute Error')
ax.set_xlabel('Severity Range')
ax.set_title('Error Distribution by Severity Level')
plt.tight_layout()
plt.show()

## Summary

In [ ]:
print("Severity Modeling Summary:")
print("- Task: Regression of severity scores (0-3 scale)")
print("- Model: EfficientNet backbone with regression head")
print("- Metrics: MAE, RMSE, R² score")
print("- Alternative: Classification approach with 4 severity levels")
print("- Error analysis helps identify severity ranges needing improvement")